# Penguins Classification - MLflow Experiment Tracking

Entrenamiento de modelos de clasificación sobre el dataset Palmer Penguins.
Se realizan 24 ejecuciones con variaciones de hiperparámetros, registradas en MLflow.

In [1]:
!pip install -q -r requirements.txt

In [2]:
import os
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from palmerpenguins import load_penguins
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sqlalchemy import create_engine

## 1. Carga y preprocesamiento de datos

In [3]:
mysql_host = os.environ.get("MYSQL_HOST", "data_db")
mysql_user = os.environ.get("MYSQL_USER", "datauser")
mysql_pass = os.environ.get("MYSQL_PASSWORD", "datapass")
mysql_db = os.environ.get("MYSQL_DATABASE", "penguins_data")
engine = create_engine(f"mysql+pymysql://{mysql_user}:{mysql_pass}@{mysql_host}:3306/{mysql_db}")

# 1a. Cargar datos raw en MySQL
df_raw = load_penguins()
df_raw.to_sql("penguins_raw", engine, if_exists="replace", index=False)
print(f"Datos raw guardados en MySQL: {len(df_raw)} filas")

# 1b. Leer raw desde MySQL, limpiar y transformar
df = pd.read_sql("SELECT * FROM penguins_raw", engine)
df = df.dropna().drop_duplicates()

species_mapping = {'Adelie': 0, 'Chinstrap': 1, 'Gentoo': 2}
df['species'] = df['species'].map(species_mapping)
df = pd.get_dummies(df)

# 1c. Guardar datos procesados en MySQL
df.to_sql("penguins_processed", engine, if_exists="replace", index=False)
print(f"Datos procesados guardados en MySQL: {len(df)} filas")

# 1d. Leer procesados desde MySQL para entrenar
df = pd.read_sql("SELECT * FROM penguins_processed", engine)
X = df.drop(columns=['species'])
y = df['species']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

Datos raw guardados en MySQL: 344 filas
Datos procesados guardados en MySQL: 333 filas
Train: 233, Val: 50, Test: 50


## 2. Configuración MLflow

In [4]:
mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "http://mlflow:5000"))
mlflow.set_experiment("penguins-classification")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

2026/03/25 00:50:07 INFO mlflow.tracking.fluent: Experiment with name 'penguins-classification' does not exist. Creating a new experiment.


Tracking URI: http://mlflow:5000


## 3. Definición de experimentos (24 configuraciones)

In [5]:
experiments = [
    # SVM - 8 variaciones
    {"model_type": "svm", "params": {"kernel": "rbf", "C": 0.1}},
    {"model_type": "svm", "params": {"kernel": "rbf", "C": 1.0}},
    {"model_type": "svm", "params": {"kernel": "rbf", "C": 10.0}},
    {"model_type": "svm", "params": {"kernel": "rbf", "C": 100.0}},
    {"model_type": "svm", "params": {"kernel": "linear", "C": 0.1}},
    {"model_type": "svm", "params": {"kernel": "linear", "C": 1.0}},
    {"model_type": "svm", "params": {"kernel": "linear", "C": 10.0}},
    {"model_type": "svm", "params": {"kernel": "poly", "C": 1.0, "degree": 3}},
    # Logistic Regression - 8 variaciones
    {"model_type": "logistic_regression", "params": {"C": 0.01, "max_iter": 1000, "solver": "lbfgs"}},
    {"model_type": "logistic_regression", "params": {"C": 0.1, "max_iter": 1000, "solver": "lbfgs"}},
    {"model_type": "logistic_regression", "params": {"C": 1.0, "max_iter": 1000, "solver": "lbfgs"}},
    {"model_type": "logistic_regression", "params": {"C": 10.0, "max_iter": 1000, "solver": "lbfgs"}},
    {"model_type": "logistic_regression", "params": {"C": 1.0, "max_iter": 1000, "solver": "saga", "penalty": "l1"}},
    {"model_type": "logistic_regression", "params": {"C": 0.1, "max_iter": 1000, "solver": "saga", "penalty": "l1"}},
    {"model_type": "logistic_regression", "params": {"C": 1.0, "max_iter": 500, "solver": "lbfgs"}},
    {"model_type": "logistic_regression", "params": {"C": 0.5, "max_iter": 2000, "solver": "lbfgs"}},
    # Random Forest - 8 variaciones
    {"model_type": "random_forest", "params": {"n_estimators": 50, "max_depth": 3, "random_state": 42}},
    {"model_type": "random_forest", "params": {"n_estimators": 100, "max_depth": 3, "random_state": 42}},
    {"model_type": "random_forest", "params": {"n_estimators": 100, "max_depth": 5, "random_state": 42}},
    {"model_type": "random_forest", "params": {"n_estimators": 100, "max_depth": 10, "random_state": 42}},
    {"model_type": "random_forest", "params": {"n_estimators": 100, "max_depth": None, "random_state": 42}},
    {"model_type": "random_forest", "params": {"n_estimators": 200, "max_depth": 5, "random_state": 42}},
    {"model_type": "random_forest", "params": {"n_estimators": 200, "max_depth": None, "random_state": 42}},
    {"model_type": "random_forest", "params": {"n_estimators": 50, "max_depth": None, "min_samples_split": 5, "random_state": 42}},
]

print(f"Total de configuraciones: {len(experiments)}")

Total de configuraciones: 24


## 4. Ejecución de experimentos

In [6]:
MODEL_CLASSES = {
    "svm": SVC,
    "logistic_regression": LogisticRegression,
    "random_forest": RandomForestClassifier,
}

results = []

for i, exp in enumerate(experiments):
    model_type = exp["model_type"]
    params = exp["params"]
    run_name = f"{model_type}_{i+1:02d}"

    with mlflow.start_run(run_name=run_name):
        mlflow.set_tag("model_type", model_type)
        mlflow.log_params(params)

        model = MODEL_CLASSES[model_type](**params)
        model.fit(X_train, y_train)

        y_val_pred = model.predict(X_val)
        y_test_pred = model.predict(X_test)

        val_accuracy = accuracy_score(y_val, y_val_pred)
        val_f1 = f1_score(y_val, y_val_pred, average="weighted")
        test_accuracy = accuracy_score(y_test, y_test_pred)
        test_f1 = f1_score(y_test, y_test_pred, average="weighted")
        test_precision = precision_score(y_test, y_test_pred, average="weighted")
        test_recall = recall_score(y_test, y_test_pred, average="weighted")

        mlflow.log_metrics({
            "val_accuracy": val_accuracy,
            "val_f1": val_f1,
            "test_accuracy": test_accuracy,
            "test_f1": test_f1,
            "test_precision": test_precision,
            "test_recall": test_recall,
        })

        mlflow.sklearn.log_model(model, artifact_path="model", registered_model_name=f"penguins-{model_type}")

        results.append({"run": run_name, "val_accuracy": val_accuracy, "test_accuracy": test_accuracy, "test_f1": test_f1})
        print(f"[{i+1:02d}/{len(experiments)}] {run_name} | val_acc={val_accuracy:.4f} | test_acc={test_accuracy:.4f} | test_f1={test_f1:.4f}")

/opt/conda/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
2026/03/25 00:50:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/25 00:50:08 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'penguins-svm'.
2026/03/25 00:50:08 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: penguins-svm, version 1
Created version '1' of model 'penguins-svm'.
/opt/conda/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is il

[01/24] svm_01 | val_acc=0.8000 | test_acc=0.6400 | test_f1=0.5458
🏃 View run svm_01 at: http://mlflow:5000/#/experiments/1/runs/71e51c5155954d8489b0a491d241a93d
🧪 View experiment at: http://mlflow:5000/#/experiments/1


2026/03/25 00:50:09 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'penguins-svm' already exists. Creating a new version of this model...
2026/03/25 00:50:09 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: penguins-svm, version 2
Created version '2' of model 'penguins-svm'.
/opt/conda/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
2026/03/25 00:50:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


[02/24] svm_02 | val_acc=0.7600 | test_acc=0.6600 | test_f1=0.5630
🏃 View run svm_02 at: http://mlflow:5000/#/experiments/1/runs/83d8aba84fd9495fbe818d5c53a5f426
🧪 View experiment at: http://mlflow:5000/#/experiments/1


2026/03/25 00:50:10 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'penguins-svm' already exists. Creating a new version of this model...
2026/03/25 00:50:10 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: penguins-svm, version 3
Created version '3' of model 'penguins-svm'.
/opt/conda/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
2026/03/25 00:50:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

2026/03/22 22:08:37 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'penguins-random_forest' already exists. Creating a new version of this model...
2026/03/22 22:08:37 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: penguins-random_forest, version 7
Created version '7' of model 'penguins-random_forest'.
2026/03/22 22:08:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


[23/24] random_forest_23 | val_acc=0.9800 | test_acc=0.9600 | test_f1=0.9591
🏃 View run random_forest_23 at: http://mlflow:5000/#/experiments/1/runs/2244175403b34a5a808f93a1d48ebfce
🧪 View experiment at: http://mlflow:5000/#/experiments/1


2026/03/22 22:08:38 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'penguins-random_forest' already exists. Creating a new version of this model...
2026/03/22 22:08:38 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: penguins-random_forest, version 8


[24/24] random_forest_24 | val_acc=0.9600 | test_acc=0.9600 | test_f1=0.9591
🏃 View run random_forest_24 at: http://mlflow:5000/#/experiments/1/runs/0a9a1b907e5b442e8fc7763de91d115c
🧪 View experiment at: http://mlflow:5000/#/experiments/1


Created version '8' of model 'penguins-random_forest'.


## 5. GridSearchCV - Búsqueda del mejor modelo

In [7]:
from sklearn.model_selection import GridSearchCV

param_grids = {
    "svm": {
        "model": SVC(),
        "params": {
            "kernel": ["rbf", "linear", "poly"],
            "C": [0.1, 1.0, 10.0, 100.0],
        }
    },
    "logistic_regression": {
        "model": LogisticRegression(max_iter=2000),
        "params": {
            "C": [0.01, 0.1, 0.5, 1.0, 10.0],
            "solver": ["lbfgs", "saga"],
        }
    },
    "random_forest": {
        "model": RandomForestClassifier(random_state=42),
        "params": {
            "n_estimators": [50, 100, 200],
            "max_depth": [3, 5, 10, None],
        }
    },
}

best_overall = {"score": 0, "model_type": None, "model": None, "params": None, "run_id": None}

for model_type, config in param_grids.items():
    run_name = f"gridsearch_{model_type}"
    with mlflow.start_run(run_name=run_name):
        mlflow.set_tag("model_type", model_type)
        mlflow.set_tag("search", "GridSearchCV")

        gs = GridSearchCV(config["model"], config["params"], cv=5, scoring="f1_weighted", n_jobs=-1)
        gs.fit(X_train, y_train)

        best_model = gs.best_estimator_
        y_test_pred = best_model.predict(X_test)

        test_acc = accuracy_score(y_test, y_test_pred)
        test_f1 = f1_score(y_test, y_test_pred, average="weighted")

        mlflow.log_params(gs.best_params_)
        mlflow.log_metrics({"cv_best_score": gs.best_score_, "test_accuracy": test_acc, "test_f1": test_f1})
        mlflow.sklearn.log_model(best_model, artifact_path="model", registered_model_name=f"penguins-{model_type}")

        print(f"[GridSearchCV] {model_type} | best_params={gs.best_params_} | cv_f1={gs.best_score_:.4f} | test_f1={test_f1:.4f}")

        if test_f1 > best_overall["score"]:
            best_overall = {
                "score": test_f1,
                "model_type": model_type,
                "model": best_model,
                "params": gs.best_params_,
                "run_id": mlflow.active_run().info.run_id,
            }

print(f"\nMejor modelo global: {best_overall['model_type']} | test_f1={best_overall['score']:.4f}")

2026/03/25 00:50:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/25 00:50:32 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'penguins-svm' already exists. Creating a new version of this model...
2026/03/25 00:50:32 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: penguins-svm, version 9
Created version '9' of model 'penguins-svm'.
/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklea

[GridSearchCV] svm | best_params={'C': 0.1, 'kernel': 'linear'} | cv_f1=0.9868 | test_f1=0.9798
🏃 View run gridsearch_svm at: http://mlflow:5000/#/experiments/1/runs/4419c66e806548d2a81630a04f51cbb3
🧪 View experiment at: http://mlflow:5000/#/experiments/1


/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which 

[GridSearchCV] logistic_regression | best_params={'C': 0.1, 'solver': 'lbfgs'} | cv_f1=0.9913 | test_f1=0.9798
🏃 View run gridsearch_logistic_regression at: http://mlflow:5000/#/experiments/1/runs/53e09b31132348caac7e697aa9719d53
🧪 View experiment at: http://mlflow:5000/#/experiments/1


2026/03/25 00:50:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/25 00:50:35 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'penguins-random_forest' already exists. Creating a new version of this model...
2026/03/25 00:50:35 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: penguins-random_forest, version 9


[GridSearchCV] random_forest | best_params={'max_depth': 5, 'n_estimators': 50} | cv_f1=0.9914 | test_f1=0.9591
🏃 View run gridsearch_random_forest at: http://mlflow:5000/#/experiments/1/runs/c0268d35a0514504aa8614ae830c991a
🧪 View experiment at: http://mlflow:5000/#/experiments/1

Mejor modelo global: svm | test_f1=0.9798


Created version '9' of model 'penguins-random_forest'.


## 6. Promover mejor modelo a producción

El mejor modelo se registra con el alias **production**, que la API de inferencia usará para cargar el modelo.

In [8]:
client = mlflow.MlflowClient()
model_name = f"penguins-{best_overall['model_type']}"

# Obtener la última versión del mejor modelo
versions = client.search_model_versions(f"name='{model_name}'")
latest_version = max(v.version for v in versions)

# Asignar alias "production" al mejor modelo
client.set_registered_model_alias(model_name, "production", latest_version)
print(f"Modelo '{model_name}' versión {latest_version} promovido a producción (alias: production)")
print(f"URI para carga: models:/{model_name}@production")

Modelo 'penguins-svm' versión 9 promovido a producción (alias: production)
URI para carga: models:/penguins-svm@production


## 7. Resumen de resultados

In [9]:
results_df = pd.DataFrame(results).sort_values("test_f1", ascending=False)
print("Top 5 modelos por test_f1:")
results_df.head()

Top 5 modelos por test_f1:


,run,val_accuracy,test_accuracy,test_f1
11,logistic_regression_12,1.0,1.00,1.000000
5,svm_06,1.0,1.00,1.000000
6,svm_07,1.0,1.00,1.000000
15,logistic_regression_16,1.0,0.98,0.979789
14,logistic_regression_15,1.0,0.98,0.979789
